# Расчёт метрик в SQL

В наличии данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности в сервисе для чтения электронных книг и прослушивания аудиокниг (в целях конфиденциальности название не приводится).

## Цели и задачи проекта

<font color='#777778'>
Цель: глубже понять поведение пользователей и найти примечательные закономерности.<br>

Задачи: рассчитать несколько метрик с помощью SQL и интерпретировать полученные результаты: понять, как пользователи взаимодействуют с платформой, какие виды контента наиболее популярны, а также выявить особенности поведения пользователей.
</font>

## Содержимое проекта

[1. Расчёт MAU авторов](#metka_1)

[2. Расчёт MAU произведений](#metka_2)
    
[3. Расчёт Retention Rate](#metka_3)

[4. Расчёт LTV](#metka_4)

[5. Расчёт средней выручки прослушанного часа](#metka_5)

[6. Выводы](#metka_6)

---

<a id='metka_1'></a>
### Расчёт MAU авторов.
MAU авторов - количество уникальных пользователей в месяц, которые читали или слушали конкретного автора.

<b>Результат:</b> имена топ-3 авторов с наибольшим MAU в ноябре и сами значения MAU, отсортированные по значению MAU в порядке убывания.

```sql
SELECT
    main_author_name,
    COUNT (DISTINCT puid) AS mau
FROM bookmate.audition as baud
JOIN bookmate.content as bcon ON baud.main_content_id = bcon.main_content_id
JOIN bookmate.author as bauth ON bcon.main_author_id = bauth.main_author_id
WHERE msk_business_dt_str >=  '2024-11-01' AND msk_business_dt_str <= '2024-11-30'
GROUP BY main_author_name
ORDER BY 2 DESC
LIMIT 3;
```

```sql
main_author_name	mau
Андрей Усачев	   7107
Лиана Шнайдер	   3338
Игорь Носов	     3063
```

, где<br>
`main_author_name` — имя автора контента;<br>
`mau` — значение MAU.

----
<a id='metka_2'></a>
### Расчёт MAU произведений

<b>Результат</b>: имена топ-3 произведений с наибольшим MAU в ноябре, а также списки жанров этих произведений, их авторов и сами значения MAU, отсортированные по значению MAU в порядке убывания.

```sql
SELECT
    main_content_name,
    published_topic_title_list,
    main_author_name,
    COUNT (DISTINCT puid) AS mau
FROM bookmate.audition as baud
JOIN bookmate.content as bcon ON baud.main_content_id = bcon.main_content_id
JOIN bookmate.author as bauth ON bcon.main_author_id = bauth.main_author_id
WHERE msk_business_dt_str >=  '2024-11-01' AND msk_business_dt_str <= '2024-11-30'
GROUP BY 1, 2, 3
ORDER BY 4 DESC
LIMIT 3;
```

```sql
main_content_name	                   published_topic_title_list	                           main_author_name   mau
Собачка Соня на даче	                ['Детская проза и поэзия', 'Аудио']	                  Андрей Усачев	  4597
Женькин клад и другие школьные рассказы ['Сказки и фольклор', 'Детская проза и поэзия', 'Аудио'] Игорь Носов        3050
Знаменитая собачка Соня	             ['Аудиоспектакли', 'Детская проза и поэзия', 'Аудио']	Андрей Усачев	  2785
```

,где<br>
`main_content_name` — название произведения, или контента;<br>
`published_topic_title_list` — список жанров контента;<br>
`main_author_name` — имя автора контента;<br>
`mau` — значение MAU.

----
<a id='metka_3'></a>
### Расчёт Retention Rate.
Команда сервиса провела рекламную кампанию, которая 2 декабря привлекла множество пользователей. Необходимо проанализировать ежедневный Retention Rate всех пользователей, которые были активны 2 декабря.

<b>Результат:</b> ежедневный Retention Rate пользователей до конца представленных данных, отсортированный по сроку жизни в порядке возрастания.

```sql
--Результат — список пользователей, которые были активны в день старта рекламной кампании
WITH total_puid AS(
    SELECT
        puid
    FROM bookmate.audition
    WHERE msk_business_dt_str = '2024-12-02'
),

retained_data AS(
    SELECT
        (a.msk_business_dt_str - '2024-12-02') AS day_since_install, -- номер дня с момента старта рекламной кампании
        COUNT(DISTINCT a.puid) AS retained_users--подсчёт уникальных пользователей из когорты, активных в каждый день
    FROM bookmate.audition AS a
--Результат — только те записи из bookmate.audition, где puid есть в исходной когорте (т. е. пользователи, которые были активны 2 декабря)
    JOIN total_puid ON a.puid = total_puid.puid
    WHERE msk_business_dt_str >= '2024-12-02'--учитываются только дни после старта рекламной кампании (включая её саму)
    GROUP BY 1 --группировка по day_since_install — для каждого дня свой счётчик
)

SELECT
    day_since_install,
    retained_users,
    ROUND(1.0 * retained_users / MAX(retained_users) OVER (), 2) AS retention_rate
FROM retained_data
ORDER BY 1;
```

```sql
day_since_install	retained_users	retention_rate
0	                4259	          1
1	                2698	          0.63
2	                2550	          0.6
3	                2421	          0.57
4	                2231	          0.52
5	                1994	          0.47
6	                2129	          0.5
7	                2287	          0.54
8	                2274	          0.53
9	                2207	          0.52
```

, где<br>
`day_since_install` — срок жизни пользователя в днях;<br>
`retained_users` — количество пользователей, которые вернулись в приложение в конкретный день;<br>
`retention_rate` — коэффициент удержания для вернувшихся пользователей по отношению к общему числу пользователей, которые установили приложение, округленный до двух знаков после запятой.

----
<a id='metka_4'></a>
### Расчёт LTV.
Подписка стоит 399 рублей в месяц. Будем считать, что пользователь приносит 399 рублей, если хотя бы раз в месяц пользуется книгами сервиса. При этом потенциально платящих, но неактивных пользователей не нужно учитывать.

<b>Результат:</b> средние LTV, за все время, для пользователей в Москве и Санкт-Петербурге.

```sql
-- Количество месяцев активности для каждого пользователя в каждом городе
WITH puid_months AS (
    SELECT
        puid,
        usage_geo_id,
        usage_geo_id_name,
        COUNT(DISTINCT DATE_TRUNC('month', msk_business_dt_str::date)) AS count_months
    FROM bookmate.audition a
    JOIN bookmate.geo g USING (usage_geo_id)
    WHERE g.usage_geo_id_name IN ('Москва', 'Санкт-Петербург')
    GROUP BY 1, 2, 3 
),

-- Общий доход по городам для всех пользователей
total_revenue AS (
    SELECT
        usage_geo_id,
        usage_geo_id_name AS city,
        SUM(399 * count_months) AS revenue
  FROM puid_months
  GROUP BY 1, 2
),

-- Количество уникальных пользователей по городам
total_puid AS (
    SELECT
        usage_geo_id,
        COUNT(DISTINCT puid) AS total_users
    FROM bookmate.audition
    GROUP BY 1
)
SELECT
  r.city,
  p.total_users,
  ROUND(r.revenue / p.total_users, 2) AS ltv
FROM total_revenue r
JOIN total_puid p USING (usage_geo_id)
ORDER BY r.city;
```

```sql
city            total_users	ltv
Москва	      16808	      764.55
Санкт-Петербург 12559	      731.82
```

, где<br>
`city` — название города или региона;<br>
`total_users` — суммарное количество пользователей в городе или регионе;<br>
`ltv` — средний LTV пользователей в городе или регионе, округленный до двух знаков после запятой.

----
<a id='metka_5'></a>
### Расчёт средней выручки прослушанного часа.
Предполагается, что в сервисе используется модель монетизации с единой подпиской. По этой причине рассчитывать средний чек транзакций нецелесообразно, ведь он будет равен стоимости подписки. Но можно рассчитать ежемесячную среднюю выручку от часа чтения или прослушивания.

<b>Результат:</b> ежемесячная средняя выручка от часа чтения или прослушивания вместе с MAU и суммой прослушанных часов с сентября по ноябрь.

```sql
SELECT
    DATE_TRUNC('month', msk_business_dt_str::date)::DATE AS month,
    COUNT (DISTINCT puid) AS mau,
    ROUND(SUM(hours), 2) AS hours,
    ROUND((COUNT (DISTINCT puid) * 399) / SUM(hours), 2) AS avg_hour_rev
FROM bookmate.audition
WHERE msk_business_dt_str >= '2024-09-01' AND msk_business_dt_str <= '2024-11-30'
GROUP BY 1;
```

```sql
month	   mau	hours   avg_hour_rev
2024-09-01  16320  105539  61.7
2024-10-01  18280  137384  53.09
2024-11-01  18594  145351  51.04
```

,где<br>
`month` — месяц активности (первое число месяца в формате YYYY-MM-DD);<br>
`mau` — значение MAU;<br>
`hours` — общее количество прослушанных часов, округленное до двух знаков после запятой;<br>
`avg_hour_rev` — средняя выручка от часа чтения или прослушивания, округленная до двух знаков после запятой.

----
<a id='metka_6'></a>
### Выводы:

- Детский контент и аудио — ключевой драйвер вовлечённости.
- Удержание требует фокуса на первую неделю после привлечения, особенно с 3 по 5 дни.
- Монетизация показывает нюансы: рост аудитории не гарантирует рост дохода на единицу активности (avg_hour_rev падает), а LTV по городам отличается умеренно, но системно.
- Для дальнейших шагов логично углубиться в когортный анализ по типам контента и устройствам.